# 06 — Bank Management & Configuration

This notebook covers bank lifecycle management:
- Creating banks with full configuration
- Updating bank settings
- Memory management (list, get, update, delete)
- Entity graph inspection
- Tag management
- Bank design patterns

In [ ]:
from hindsight_client import Hindsight

HINDSIGHT_URL = "http://localhost:8888"
BANK_ID = "tutorial-banks"

client = Hindsight(base_url=HINDSIGHT_URL)

print("Client ready.")

## 1. Creating a Fully-Configured Bank

Let's create a bank with every configuration option.

In [ ]:
client.create_bank(
    bank_id=BANK_ID,
    name="Fully Configured Bank",
    
    # === Reflect Configuration ===
    reflect_mission="""
    I am a senior software engineer assistant.
    I track project context, architectural decisions, and user preferences.
    I prioritize pragmatic solutions that balance velocity with quality.
    When uncertain, I seek clarification rather than assume.
    """,
    
    # Soft personality traits
    disposition={
        "skepticism": 3,    # Balanced — trust but verify
        "literalism": 2,    # Slightly flexible — read between lines
        "empathy": 3        # Balanced — consider human factors
    },
    
    # === Retain Configuration ===
    retain_mission="Extract all technical details: languages, frameworks, versions, architecture patterns.",
    retain_extraction_mode="detailed",  # More granular fact extraction
    retain_chunk_size=3000,             # Characters per chunk
    
    # === Observation Configuration ===
    enable_observations=True,           # Auto-consolidate related facts
    
    # Hard rules (not set here but can be added via update)
    # directives=[...]  # Add via update_bank()
)
print(f"✓ Created bank '{BANK_ID}' with full configuration")

## 2. Listing All Banks

In [ ]:
banks = client.list_banks()

print(f"Total banks: {len(banks)}\n")
for bank in banks:
    print(f"  ID: {bank.id}")
    print(f"  Name: {bank.name}")
    print(f"  Facts: {getattr(bank, 'fact_count', 'N/A')}")
    print(f"  Last document: {getattr(bank, 'last_document_at', 'N/A')}")
    print()

## 3. Updating a Bank

In [ ]:
# Update the mission — partial update, other settings preserved
client.update_bank(
    bank_id=BANK_ID,
    reflect_mission="""
    UPDATED: I am a senior FULL-STACK engineer assistant.
    I now also track frontend preferences and UI/UX decisions.
    """
)
print("✓ Updated mission (other settings preserved)")

# Switch to concise extraction mode
client.update_bank(
    bank_id=BANK_ID,
    retain_extraction_mode="concise"
)
print("✓ Switched to concise extraction mode")

## 4. Bank Statistics

In [ ]:
# First, add some data
for i in range(5):
    client.retain(
        bank_id=BANK_ID,
        content=f"Test fact #{i}: The team uses Python for backend services.",
        context="test"
    )

stats = client.get_bank_stats(bank_id=BANK_ID)

print("=== Bank Statistics ===")
for key in ['total_memories', 'world_facts', 'experience_facts', 
            'observations', 'entity_count', 'edge_count',
            'pending_operations', 'failed_operations',
            'last_consolidation_time']:
    val = stats.get(key, 'N/A')
    print(f"  {key}: {val}")

## 5. Memory Management

CRUD operations on individual memories.

In [ ]:
# List memories with filtering
all_memories = client.list_memories(
    bank_id=BANK_ID,
    limit=10
)
print(f"Total memories: {len(all_memories) if isinstance(all_memories, list) else 'N/A'}")

# If list returned, show first few
if isinstance(all_memories, list) and all_memories:
    for mem in all_memories[:3]:
        mem_id = getattr(mem, 'id', '?')
        mem_text = getattr(mem, 'text', str(mem))[:100]
        mem_type = getattr(mem, 'type', '?')
        print(f"  [{mem_type}] {mem_id}: {mem_text}...")

In [ ]:
# Text search in memories
alice_results = client.list_memories(
    bank_id=BANK_ID,
    search_query="backend",
    limit=10
)

if isinstance(alice_results, list):
    print(f"Memories matching 'backend': {len(alice_results)}")
    for mem in alice_results:
        print(f"  • {getattr(mem, 'text', str(mem))[:100]}...")

In [ ]:
# Curate a memory — mark it as invalid when information is outdated
# First, let's get a memory ID from our list
all_memories = client.list_memories(bank_id=BANK_ID, limit=1, type="world")

if isinstance(all_memories, list) and all_memories:
    mem = all_memories[0]
    mem_id = getattr(mem, 'id', None)
    
    if mem_id:
        # Invalidate it
        client.update_memory(
            bank_id=BANK_ID,
            memory_id=mem_id,
            state="invalidated",
            reason="Superseded by newer information"
        )
        print(f"✓ Invalidated memory {mem_id}")
        print(f"  Reason: Superseded by newer information")
        print(f"  This triggers re-consolidation of dependent observations.")
    else:
        print("Could not get memory ID")
else:
    print("No world facts to curate yet.")

## 6. Entity Graph Inspection

In [ ]:
# Inspect the entity graph
graph = client.get_graph(bank_id=BANK_ID)

print("=== Entity Graph ===")
if hasattr(graph, 'nodes'):
    print(f"Entities (nodes): {len(graph.nodes)}")
    for node in graph.nodes[:10]:
        name = getattr(node, 'name', str(node))
        ntype = getattr(node, 'type', '?')
        print(f"  • {name} ({ntype})")

if hasattr(graph, 'edges'):
    print(f"\nRelationships (edges): {len(graph.edges)}")
    for edge in graph.edges[:10]:
        src = getattr(edge, 'source', '?')
        rel = getattr(edge, 'relationship', getattr(edge, 'type', '?'))
        tgt = getattr(edge, 'target', '?')
        print(f"  • {src} --{rel}--> {tgt}")

## 7. Tags

In [ ]:
# List tags in the bank
tags = client.list_tags(bank_id=BANK_ID)

print("=== Tags ===")
if isinstance(tags, list):
    for tag in tags:
        name = getattr(tag, 'name', str(tag))
        count = getattr(tag, 'count', '?')
        print(f"  • {name} ({count})")
elif isinstance(tags, dict):
    for name, count in tags.items():
        print(f"  • {name} ({count})")

## 8. Memory Time-Series

In [ ]:
# View ingestion pattern over time
timeseries = client.get_memory_timeseries(
    bank_id=BANK_ID,
    period="7d"
)
print("=== Memory Ingestion (7 days) ===")
if isinstance(timeseries, list):
    for bucket in timeseries:
        ts = getattr(bucket, 'timestamp', str(bucket))
        count = getattr(bucket, 'count', '?')
        print(f"  {ts}: {count} memories")

## 9. Bank Design Patterns

In [ ]:
# Pattern 1: One bank per user
class PerUserBankStrategy:
    """Creates and manages one bank per user."""
    
    def __init__(self, client):
        self.client = client
    
    def get_or_create_bank(self, user_id, user_name=""):
        bank_id = f"user-{user_id}"
        try:
            self.client.get_bank_stats(bank_id=bank_id)
        except Exception:
            self.client.create_bank(
                bank_id=bank_id,
                name=f"User: {user_name or user_id}",
                reflect_mission=f"Personal assistant for {user_name or user_id}."
            )
        return bank_id

# Pattern 2: Project-based banks
class ProjectBankStrategy:
    """One bank per project, with shared knowledge injection."""
    
    def __init__(self, client):
        self.client = client
    
    def setup_project(self, project_id, project_info):
        bank_id = f"project-{project_id}"
        
        self.client.create_bank(
            bank_id=bank_id,
            name=f"Project: {project_info['name']}",
            reflect_mission=f"""
            Project context for {project_info['name']}.
            Stack: {project_info.get('stack', 'N/A')}
            Team: {project_info.get('team_size', '?')} engineers
            """
        )
        
        # Seed with project context
        if 'context' in project_info:
            self.client.retain_batch(
                bank_id=bank_id,
                items=[{"content": c, "context": "project-context"} 
                       for c in project_info['context']]
            )
        
        return bank_id

print("✓ Pattern classes defined")
print("  PerUserBankStrategy: one bank per user ID")
print("  ProjectBankStrategy: one bank per project with seeded context")

## 10. Cleanup

In [ ]:
# Uncomment to delete
# client.delete_bank(bank_id=BANK_ID)
# WARNING: This is IRREVERSIBLE — deletes all memories, entities, and observations
print("Bank preserved. Delete manually if needed.")

## Summary

You learned:
1. **Creating banks** with full configuration (mission, retain mode, disposition)
2. **Listing and updating** banks
3. **Bank statistics** — monitoring memory counts, pending operations
4. **Memory management** — listing, searching, curating (invalidating) memories
5. **Entity graph** — inspecting the knowledge graph
6. **Tags and timeseries** — organizing and monitoring memory growth
7. **Design patterns** — per-user banks, project banks

**Next:** [07_mental_models.ipynb](07_mental_models.ipynb) — mental models and observations